In [1]:
import numpy as np 
import soundfile as sf
import librosa
import pandas as pd 

from pathlib import Path
import pickle 
import IPython.display as ipd
import sys 
sys.path.append('../../datasets/spatial_audio_pipeline/spatial_audio_util/')
import util_audio 
import soxr
from tqdm.auto import tqdm
import librosa

# Make stimuli for 2024 mono unfamiliar language distractor experiment
___
## Wanted Conditions:
### SNRs:
-9, -6, -3, 0, 3, inf 
### Distractor conditions:
- 1-talker
___
Will be using foregrounds and cues from manifest:
`/om/user/imgriff/datasets/human_word_rec_SWC_2023/unique_cue_and_target_manifest_for_human_expmnt.pdpkl`

English language distractors will come from same set as SWC experiment. Use conditions 30-34 and 40 (1-distractor at wanted SNRs) 

Cut distractors using Chinese mandarin and dutch:
Source from common voice:
- zh-CN is Chinese mandarin subset `/om2/data/public/mozilla-CommonVoice-9.0/cv-corpus-9.0-2022-04-27/zh-CN` 
- nl is Dutch subset `/om2/data/public/mozilla-CommonVoice-9.0/cv-corpus-9.0-2022-04-27/nl` 

Will be using dev split from each language experiment stimuli
___
Stim will be moved to `/om/user/imgriff/datasets/human_distractor_language_2024`
___

### Def audio read fn

## Load and prep distractor manifests

#### Get max n targets from set of foregrounds

In [2]:
stim_out_path = Path('/om/user/imgriff/datasets/human_distractor_language_2024')


In [4]:
stim_out_path = Path('/om/user/imgriff/datasets/human_distractor_language_2024')
stim_out_path.mkdir(parents=True, exist_ok=True)


manifest = pd.read_pickle('/om2/user/imgriff/datasets/spatial_audio_pipeline/assets/human_attn_experiment_v00/full_eval_trial_manifest_new_fnames.pdpkl')
# manifest = pd.read_pickle('/om/user/imgriff/datasets/human_word_rec_SWC_2023/unique_cue_and_target_manifest_for_human_expmnt.pdpkl')
# manifest = pd.read_pickle("/om/user/imgriff/datasets/human_word_rec_SWC_2023/sounds/condition_33/manifest_w_transcripts.pdpkl")

n_targets = manifest.shape[0]
print(f"{n_targets} targets")

print(manifest.gender.value_counts())
n_per_gender = manifest.gender.value_counts().values[0]

SR = 44100


1440 targets
female    720
male      720
Name: gender, dtype: int64


### Load and cut Mandarin subset


In [19]:
zh_df = pd.read_csv('/om2/data/public/mozilla-CommonVoice-9.0/cv-corpus-9.0-2022-04-27/zh-CN/dev.tsv', sep="\t")
zh_df = zh_df[zh_df.gender.isin(['male', 'female'])]
print(zh_df.gender.value_counts())
zh_df = zh_df.groupby('gender').sample(n_per_gender, replace=False, random_state=0).reset_index(drop=True)
print(zh_df.gender.value_counts())


male      4373
female    1030
Name: gender, dtype: int64
female    720
male      720
Name: gender, dtype: int64


#### Cut and track output fnames

In [4]:
import warnings
warnings.filterwarnings('ignore')

In [10]:
out_dir = stim_out_path / "zh_distractor_excerpts"
out_dir.mkdir(parents=True, exist_ok=True)

cv_stim_dir = Path("/om2/data/public/mozilla-CommonVoice-9.0/cv-corpus-9.0-2022-04-27") 
zh_stim_dir = cv_stim_dir / "zh-CN/clips"

sig_dur = 2 # in seconds
ramp_dur = 10 
win_size = int(sig_dur * SR)
dBSPL = 60

## Write and track new names 

new_out_names = []
for f_path in tqdm(zh_df.path.values):
    # load audio 
    y, _ = librosa.load(zh_stim_dir / f_path, sr = SR)
    y = util_audio.pad_or_trim_to_len(y, win_size)
    y = np.nan_to_num(y.astype(np.float32), nan=0.0, posinf=0.0, neginf=0.0)
    y = util_audio.set_dBSPL(y, dBSPL)
    y = util_audio.ramp_hann(y, ramp_dur, samplerate=SR)
    out_name = out_dir / f"{f_path.split('.')[0]}_3_sec.wav"
    new_out_names.append(out_name)
    #write as wav
    sf.write(out_name, y, samplerate=SR)



  0%|          | 0/1440 [00:00<?, ?it/s]

In [11]:
zh_df['new_excerpt_path'] = new_out_names
zh_df['new_excerpt_SR'] = SR
zh_df['new_excerpt_dur'] = sig_dur


#### Save parent manifest

In [12]:
zh_out_name = stim_out_path / 'commonvoice_zh_chinese_distractor_meta_from_dev_split.pdpkl'
zh_df.to_pickle(zh_out_name)

In [5]:
### Reload if crashed 
zh_out_name = stim_out_path / 'commonvoice_zh_chinese_distractor_meta_from_dev_split.pdpkl'
zh_df = pd.read_pickle(zh_out_name)

In [16]:
ipd.Audio(zh_df.new_excerpt_path[100])

### Load and cut Dutch subset


In [20]:
nl_df = pd.read_csv('/om2/data/public/mozilla-CommonVoice-9.0/cv-corpus-9.0-2022-04-27/nl/dev.tsv', sep="\t")
nl_df = nl_df[nl_df.gender.isin(['male', 'female'])]
print(nl_df.gender.value_counts())
nl_df = nl_df.groupby('gender').sample(n_per_gender, replace=False, random_state=0).reset_index(drop=True)
print(nl_df.gender.value_counts())


male      5282
female    1786
Name: gender, dtype: int64
female    720
male      720
Name: gender, dtype: int64


#### Cut and track output fnames

In [22]:
out_dir = stim_out_path / "nl_distractor_excerpts"
out_dir.mkdir(parents=True, exist_ok=True)

cv_stim_dir = Path("/om2/data/public/mozilla-CommonVoice-9.0/cv-corpus-9.0-2022-04-27") 
nl_stim_dir = cv_stim_dir / "nl/clips"

sig_dur = 2 # in seconds
ramp_dur = 10 
win_size = int(sig_dur * SR)
dBSPL = 60

## Write and track new names 

new_out_names = []
for f_path in tqdm(nl_df.path.values):
    # load audio 
    y, _ = librosa.load(nl_stim_dir / f_path, sr = SR)
    y = util_audio.pad_or_trim_to_len(y, win_size)
    y = np.nan_to_num(y.astype(np.float32), nan=0.0, posinf=0.0, neginf=0.0)
    y = util_audio.set_dBSPL(y, dBSPL)
    y = util_audio.ramp_hann(y, ramp_dur, samplerate=SR)
    out_name = out_dir / f"{f_path.split('.')[0]}_3_sec.wav"
    new_out_names.append(out_name)
    #write as wav
    sf.write(out_name, y, samplerate=SR)



nl_df['new_excerpt_path'] = new_out_names
nl_df['new_excerpt_SR'] = SR
nl_df['new_excerpt_dur'] = sig_dur


  0%|          | 0/1440 [00:00<?, ?it/s]

#### Save parent manifest with new paths

In [25]:
nl_out_name = stim_out_path / 'commonvoice_nl_dutch_distractor_meta_from_dev_split.pdpkl'
nl_df.to_pickle(nl_out_name)

In [6]:
### Reload if crashed 
nl_out_name = stim_out_path / 'commonvoice_nl_dutch_distractor_meta_from_dev_split.pdpkl'
nl_df = pd.read_pickle(nl_out_name)

In [30]:
ipd.Audio(nl_df.new_excerpt_path[10])

#### Full English target, cue, and distractor signals included in full manifest

In [7]:
np.unique([librosa.get_duration(filename=fn) for fn in manifest.src_fn.values])

array([2.])

In [8]:
np.unique([librosa.get_duration(filename=fn) for fn in manifest.cue_src_fn.values])

array([2.])

In [9]:
np.unique([librosa.get_duration(filename=fn) for fn in manifest.distractor_src_fn.values])

array([2.])

## Add different language distractor fnames to manifest 

### Add mandarin distractors

In [10]:
manifest.loc[manifest.distractor_gender == 'female', ['zh_distractor_client_id', 'zh_distractor_gender', 'zh_distractor_src_fn'] ] =  zh_df.loc[zh_df.gender == 'female', ['client_id', 'gender', 'new_excerpt_path']].values
manifest.loc[manifest.distractor_gender == 'male', ['zh_distractor_client_id', 'zh_distractor_gender', 'zh_distractor_src_fn'] ] =  zh_df.loc[zh_df.gender == 'male', ['client_id', 'gender', 'new_excerpt_path']].values

### Add dutch distractors

In [11]:
manifest.loc[manifest.distractor_gender == 'female', ['nl_distractor_client_id', 'nl_distractor_gender', 'nl_distractor_src_fn'] ] =  nl_df.loc[nl_df.gender == 'female', ['client_id', 'gender', 'new_excerpt_path']].values
manifest.loc[manifest.distractor_gender == 'male', ['nl_distractor_client_id', 'nl_distractor_gender', 'nl_distractor_src_fn'] ] =  nl_df.loc[nl_df.gender == 'male', ['client_id', 'gender', 'new_excerpt_path']].values

##### Indicate common voice is source corpus for langauge distractors

In [19]:
manifest['nl_distractor_corpus'] = 'cv'
manifest['zh_distractor_corpus'] = 'cv' 

#### Make sure distractor sex still matches distribution

In [15]:
dist_sex_manifest = manifest[['distractor_gender','nl_distractor_gender', 'zh_distractor_gender']]
dist_sex_manifest.eq(dist_sex_manifest.iloc[:, 0], axis=0).all(1).all()

True

### Drop unecessary columns and save global stim manifest

In [20]:
manifest.columns

Index(['distractor_client_id', 'distractor_clip_dur_in_s',
       'distractor_clip_end_in_s', 'distractor_clip_start_in_s',
       'distractor_corpus', 'distractor_gender', 'distractor_gender_int',
       'distractor_split', 'distractor_split_int', 'distractor_sr',
       'distractor_src_fn', 'distractor_total_file_duration_in_s',
       'distractor_word', 'src_ix', 'client_id', 'clip_dur_in_s',
       'clip_end_in_s', 'clip_start_in_s', 'corpus', 'gender', 'gender_int',
       'split', 'split_int', 'sr', 'src_fn', 'total_file_duration_in_s',
       'word', 'cue_word', 'cue_src_ix', 'cue_client_id', 'cue_src_fn',
       'cue_clip_start_in_s', 'cue_clip_end_in_s', 'gender_cond_td',
       'cue_clip_dur_in_s', 'zh_distractor_client_id', 'zh_distractor_gender',
       'zh_distractor_src_fn', 'nl_distractor_client_id',
       'nl_distractor_gender', 'nl_distractor_src_fn', 'nl_distractor_corpus',
       'zh_distractor_corpus'],
      dtype='object')

In [30]:
wanted_columns = ['distractor_client_id', 
       'distractor_corpus', 'distractor_gender', 'distractor_gender_int',
       'distractor_sr',
       'distractor_src_fn', 
       'distractor_word', 'src_ix', 'client_id', 
       'corpus', 'gender', 'gender_int',
       'sr', 'src_fn', 
       'word', 'cue_word', 'cue_src_ix', 'cue_client_id', 'cue_src_fn',
       'gender_cond_td',
       'zh_distractor_client_id', 'zh_distractor_gender',
       'zh_distractor_src_fn', 'nl_distractor_client_id',
       'nl_distractor_gender', 'nl_distractor_src_fn', 'nl_distractor_corpus',
       'zh_distractor_corpus']

final_manifest = manifest[wanted_columns]

final_manifest['signal_len_s'] = 2

final_manifest.head()

/tmp/ipykernel_3921507/2694516909.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final_manifest['signal_len_s'] = 2


,distractor_client_id,distractor_corpus,distractor_gender,distractor_gender_int,distractor_sr,distractor_src_fn,distractor_word,src_ix,client_id,corpus,...,gender_cond_td,zh_distractor_client_id,zh_distractor_gender,zh_distractor_src_fn,nl_distractor_client_id,nl_distractor_gender,nl_distractor_src_fn,nl_distractor_corpus,zh_distractor_corpus,signal_len_s
0,popularoutcast,swc,female,0,44100,/om/user/imgriff/datasets/spatial_audio_pipeli...,either,992601,1904-cc,swc,...,female_female,9c307658ff5206154edb3bf836398e90c07bc836a36619...,female,/om/user/imgriff/datasets/human_distractor_lan...,45cd595afd01e386500dfeff20804f2b8cc3d0fc0382cb...,female,/om/user/imgriff/datasets/human_distractor_lan...,cv,cv,2
1,matthewdgonzalez,swc,male,1,44100,/om/user/imgriff/datasets/spatial_audio_pipeli...,seven,992601,1904-cc,swc,...,female_male,4cc2f0ae39b1da188ff383559e097eaaa3a902f990d95e...,male,/om/user/imgriff/datasets/human_distractor_lan...,3108ab73b3f900eacdc24c1d12c7f765b3fe0705a6c624...,male,/om/user/imgriff/datasets/human_distractor_lan...,cv,cv,2
2,flyingtoaster,swc,female,0,44100,/om/user/imgriff/datasets/spatial_audio_pipeli...,were,993029,1904-cc,swc,...,female_female,04776675c17170276c1037565d816677753f4bb875be70...,female,/om/user/imgriff/datasets/human_distractor_lan...,493a95c46abb885f230c8ec86e26ab3403148b6410ae89...,female,/om/user/imgriff/datasets/human_distractor_lan...,cv,cv,2
3,warmvoiceover,swc,male,1,44100,/om/user/imgriff/datasets/spatial_audio_pipeli...,created,993029,1904-cc,swc,...,female_male,7a53c6a802335ae7a86f562d0d6e23efdc4665cb5ddf06...,male,/om/user/imgriff/datasets/human_distractor_lan...,55edb0faa36b1882a71261e93de75db781f61128b3876c...,male,/om/user/imgriff/datasets/human_distractor_lan...,cv,cv,2
4,popularoutcast,swc,female,0,44100,/om/user/imgriff/datasets/spatial_audio_pipeli...,unknown,1003750,99of9-toby-hudson,swc,...,male_female,40cc442a2614c3c8e232bbc17f7cf4c9024feb6e12208c...,female,/om/user/imgriff/datasets/human_distractor_lan...,52112f4c966d9e2246d170b0d2b6ee6886999b9827df6e...,female,/om/user/imgriff/datasets/human_distractor_lan...,cv,cv,2


In [10]:
word_dict = pickle.load(open("/om2/user/imgriff/datasets/commonvoice_9/en/cv_800_word_label_to_int_dict.pkl", 'rb'))


In [32]:

final_manifest['word_int'] = final_manifest['word'].map(word_dict)
final_manifest['distractor_word_int'] = final_manifest['distractor_word'].map(word_dict)

/tmp/ipykernel_3921507/885237323.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final_manifest['word_int'] = final_manifest['word'].map(word_dict)
/tmp/ipykernel_3921507/885237323.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final_manifest['distractor_word_int'] = final_manifest['distractor_word'].map(word_dict)


In [34]:
final_manifest['word_int'].isna().any()

False

### Write global stim manifest

In [37]:
final_out_name = stim_out_path / 'final_stim_manifest_w_cue_tg_lang_dists.pdpkl'
final_manifest.to_pickle(final_out_name)

In [2]:
stim_out_path = Path('/om/user/imgriff/datasets/human_distractor_language_2024')
final_out_name = stim_out_path / 'final_stim_manifest_w_cue_tg_lang_dists.pdpkl'

final_manifest = pd.read_pickle(final_out_name)

In [26]:
final_manifest.columns

Index(['distractor_client_id', 'distractor_corpus', 'distractor_gender',
       'distractor_gender_int', 'distractor_sr', 'distractor_src_fn',
       'distractor_word', 'src_ix', 'client_id', 'corpus', 'gender',
       'gender_int', 'sr', 'src_fn', 'word', 'cue_word', 'cue_src_ix',
       'cue_client_id', 'cue_src_fn', 'gender_cond_td',
       'zh_distractor_client_id', 'zh_distractor_gender',
       'zh_distractor_src_fn', 'nl_distractor_client_id',
       'nl_distractor_gender', 'nl_distractor_src_fn', 'nl_distractor_corpus',
       'zh_distractor_corpus', 'signal_len_s', 'word_int',
       'distractor_word_int'],
      dtype='object')

In [30]:
final_manifest[['cue_src_fn', 'src_fn']].values[0]

array(['/om/user/imgriff/datasets/spatial_audio_pipeline/assets/human_attn_experiment_v00/cue_excerpts/dramatic_1904-cc.wav',
       '/om/user/imgriff/datasets/spatial_audio_pipeline/assets/human_attn_experiment_v00/target_excerpts/these_1904-cc.wav'],
      dtype=object)

### Manually handle contractions 

In [22]:
word_dict['were'] = word_dict["we're"]
word_dict['dont'] = word_dict["don't"]
word_dict['didnt'] = word_dict["didn't"]
word_dict['doesnt'] = word_dict["doesn't"]


In [23]:
final_manifest['distractor_word_int'] = final_manifest['distractor_word'].map(word_dict)

In [24]:
final_manifest['distractor_word_int'].isna().any()

False

In [25]:
final_out_name = stim_out_path / 'final_stim_manifest_w_cue_tg_lang_dists.pdpkl'
final_manifest.to_pickle(final_out_name)

## Make and write audio for human experiment 

Will use same set of curated unique words as in SWC and Popham style experiments 

In [22]:
stim_out_path = Path('/om/user/imgriff/datasets/human_distractor_language_2024')
final_out_name = stim_out_path / 'final_stim_manifest_w_cue_tg_lang_dists.pdpkl'

final_manifest = pd.read_pickle(final_out_name)

manifest = pd.read_pickle("/om/user/imgriff/datasets/human_word_rec_SWC_2023/sounds/condition_33/manifest_w_transcripts.pdpkl")


In [23]:
final_manifest.columns

Index(['distractor_client_id', 'distractor_corpus', 'distractor_gender',
       'distractor_gender_int', 'distractor_sr', 'distractor_src_fn',
       'distractor_word', 'src_ix', 'client_id', 'corpus', 'gender',
       'gender_int', 'sr', 'src_fn', 'word', 'cue_word', 'cue_src_ix',
       'cue_client_id', 'cue_src_fn', 'gender_cond_td',
       'zh_distractor_client_id', 'zh_distractor_gender',
       'zh_distractor_src_fn', 'nl_distractor_client_id',
       'nl_distractor_gender', 'nl_distractor_src_fn', 'nl_distractor_corpus',
       'zh_distractor_corpus', 'signal_len_s', 'word_int',
       'distractor_word_int'],
      dtype='object')

In [24]:
manifest.columns

Index(['target_sr', 'experiment_key_target_word_ix', 'cue_sr', 'target_fn',
       'cue_fn', 'word', 'word_int', 'condition', 'snr', 'src_ix', 'client_id',
       'target_gender', 'target_f0', 'distractor_fn', 'distractor_f0',
       'distractor_gender', 'distractor_word', 'mixture_fn',
       'target_transcripts', 'distractor_transcripts'],
      dtype='object')

In [32]:
human_exp_manifest = pd.merge(manifest,
                            final_manifest[['src_fn', 'cue_src_fn', 'distractor_src_fn', 'word', 'distractor_word',
                                            'zh_distractor_client_id', 'zh_distractor_gender',
                                            'zh_distractor_src_fn', 'nl_distractor_client_id',
                                            'nl_distractor_gender', 'nl_distractor_src_fn', 'nl_distractor_corpus',
                                            'zh_distractor_corpus']],
                        left_on=['target_fn', 'cue_fn', 'distractor_fn', 'word', 'distractor_word'],
                        right_on=['src_fn', 'cue_src_fn', 'distractor_src_fn', 'word', 'distractor_word'],
                        how='left').drop(columns=['src_fn', 'cue_src_fn', 'distractor_src_fn'])


In [33]:
human_exp_manifest.columns

Index(['target_sr', 'experiment_key_target_word_ix', 'cue_sr', 'target_fn',
       'cue_fn', 'word', 'word_int', 'condition', 'snr', 'src_ix', 'client_id',
       'target_gender', 'target_f0', 'distractor_fn', 'distractor_f0',
       'distractor_gender', 'distractor_word', 'mixture_fn',
       'target_transcripts', 'distractor_transcripts',
       'zh_distractor_client_id', 'zh_distractor_gender',
       'zh_distractor_src_fn', 'nl_distractor_client_id',
       'nl_distractor_gender', 'nl_distractor_src_fn', 'nl_distractor_corpus',
       'zh_distractor_corpus'],
      dtype='object')

In [34]:
human_exp_manifest.to_pickle(stim_out_path / "human_expmt_manifest_w_transcripts.pdpkl" )

### Prep condition list

In [13]:
# Make condition dict 
import itertools 
import pickle 
import numpy as np 
snrs = np.arange(-9, 4, 3).tolist() # -9, -6, -3, 0, 3

conditions = ['english_1-distractor',
              'dutch_1-distractor',
              'mandarin_1-distractor']

condition_pairs = list(itertools.product(conditions, snrs))
condition_pairs.append(('SILENCE', 'inf'))
cond_ix_dict = {ix:cond for ix, cond in enumerate(condition_pairs)}

out_name = stim_out_path / "human_distractor_language_cond_map.pkl"
# write condition dict as pickle 
with open(out_name, 'wb') as f:
    pickle.dump(cond_ix_dict, f)


In [19]:
stim_out_path

PosixPath('/om/user/imgriff/datasets/human_distractor_language_2024')

In [6]:
cond_ix_dict

{0: ('english_1-distractor', -9),
 1: ('english_1-distractor', -6),
 2: ('english_1-distractor', -3),
 3: ('english_1-distractor', 0),
 4: ('english_1-distractor', 3),
 5: ('dutch_1-distractor', -9),
 6: ('dutch_1-distractor', -6),
 7: ('dutch_1-distractor', -3),
 8: ('dutch_1-distractor', 0),
 9: ('dutch_1-distractor', 3),
 10: ('mandarin_1-distractor', -9),
 11: ('mandarin_1-distractor', -6),
 12: ('mandarin_1-distractor', -3),
 13: ('mandarin_1-distractor', 0),
 14: ('mandarin_1-distractor', 3),
 15: ('SILENCE', 'inf')}

### Make target word key 


In [37]:
import json
import pickle 


exmpt_word_dict = {i:w for i,w in human_exp_manifest.word.items()}
exmpt_word_dict

# save as pickle
out_name = stim_out_path /  "human_distractor_language_word_key.pkl"
with open(out_name, 'wb') as f:
    pickle.dump(exmpt_word_dict, f)



words = list(exmpt_word_dict.values())
# save as json 
out_name = stim_out_path /  "human_distractor_language_word_key.js"
with open(out_name, 'w') as f:
    json.dump({"dictionary":words}, f)



In [38]:
## Make word Key .js from pickle of word dict
import IPython.display as ipd


In [41]:
ipd.Audio("/om/user/imgriff/datasets/human_distractor_language_2024/sounds/condition_00/105.wav")

In [40]:
human_exp_manifest.loc[105, 'word']

'english'

## Make sure stimuli generation worked 

In [42]:
paths = Path('/om/user/imgriff/datasets/human_distractor_language_2024/sounds/').rglob('*/*.wav')

In [44]:
import librosa

In [47]:
bad_paths = []
for path in paths:
    try:
        librosa.get_duration(path=path)
    except Exception as e:
        print(e)
        bad_paths.append(path)